## 2. Preprocesado

In [64]:
!pip install beautifulsoup4
!pip install spacy
!pip install nltk

In [67]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from spacy.lang.en.stop_words import STOP_WORDS
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')  # Open Multilingual Wordnet (necesario para lematización)
from nltk.stem import WordNetLemmatizer
import unicodedata
import re

[nltk_data] Downloading package wordnet to /Users/maru/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/maru/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/maru/nltk_data...


In [70]:
# Cargar DataFrame desde el archivo guardado en el notebook 1
df = pd.read_pickle('Data/df_beauty_balanced.pkl')
print(f"DataFrame cargado: {len(df)} reviews")
df.head()

DataFrame cargado: 6000 reviews


,review,sentiment,sentiment_label
0,"This product is oil based, weird, and definite...",5.0,0
1,I was shown an expensive one like this in Alab...,5.0,0
2,I used to buy and use the original product. I ...,1.0,1
3,This product was advertised as brand new. Ho...,1.0,1
4,this was a new edition to our daughter's body ...,5.0,0


### Usamos primero BeautifulSoup antes de hacer split por palabras, para hacer la limpieza de etiquetas HTLM.

In [71]:
def remove_html_tags(text):
     #Elimina etiquetas HTML del texto
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ')

# Aplicar a la columna review original antes de procesar
df['review'] = df['review'].apply(remove_html_tags)
print("Etiquetas HTML eliminadas")
df.head()

Etiquetas HTML eliminadas


,review,sentiment,sentiment_label
0,"This product is oil based, weird, and definite...",5.0,0
1,I was shown an expensive one like this in Alab...,5.0,0
2,I used to buy and use the original product. I ...,1.0,1
3,This product was advertised as brand new. Ho...,1.0,1
4,this was a new edition to our daughter's body ...,5.0,0


### Convertimos a minusculas y separamos por palabras

In [72]:
df['review_process'] = df['review'].str.lower().str.split()
df.head()

,review,sentiment,sentiment_label,review_process
0,"This product is oil based, weird, and definite...",5.0,0,"[this, product, is, oil, based,, weird,, and, ..."
1,I was shown an expensive one like this in Alab...,5.0,0,"[i, was, shown, an, expensive, one, like, this..."
2,I used to buy and use the original product. I ...,1.0,1,"[i, used, to, buy, and, use, the, original, pr..."
3,This product was advertised as brand new. Ho...,1.0,1,"[this, product, was, advertised, as, brand, ne..."
4,this was a new edition to our daughter's body ...,5.0,0,"[this, was, a, new, edition, to, our, daughter..."


### Eliminación stopwords

Utilizo la libreria de spacy, de la función voy a sacar palabras que puedan ser relevantes para un analisis de sentimiento. 

In [73]:
print(list(STOP_WORDS)[:20])

['several', '’s', 'by', 'on', 'what', 'both', 'become', 'such', 'were', 'hereafter', 'every', 'everything', 'there', 'within', 'amount', 'in', 'part', 'ten', 'as', 'seeming']


In [74]:
len(list(STOP_WORDS))

326

In [75]:
# Visualizar las stopwords
print(f"Total de stopwords: {len(STOP_WORDS)}\n")
print("Lista de stopwords:")
print(sorted(list(STOP_WORDS)))

Total de stopwords: 326

Lista de stopwords:
["'d", "'ll", "'m", "'re", "'s", "'ve", 'a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although', 'always', 'am', 'among', 'amongst', 'amount', 'an', 'and', 'another', 'any', 'anyhow', 'anyone', 'anything', 'anyway', 'anywhere', 'are', 'around', 'as', 'at', 'back', 'be', 'became', 'because', 'become', 'becomes', 'becoming', 'been', 'before', 'beforehand', 'behind', 'being', 'below', 'beside', 'besides', 'between', 'beyond', 'both', 'bottom', 'but', 'by', 'ca', 'call', 'can', 'cannot', 'could', 'did', 'do', 'does', 'doing', 'done', 'down', 'due', 'during', 'each', 'eight', 'either', 'eleven', 'else', 'elsewhere', 'empty', 'enough', 'even', 'ever', 'every', 'everyone', 'everything', 'everywhere', 'except', 'few', 'fifteen', 'fifty', 'first', 'five', 'for', 'former', 'formerly', 'forty', 'four', 'from', 'front', 'full', 'further', 'get', 'give', 'go', 'had', 'ha

In [76]:
# Identificar stopwords que pueden influir en el análisis de sentimiento
sentiment_relevant_stopwords = [
    'not', 'no', 'nor', 'never', 'neither', 'nobody', 'nothing', 'nowhere',
    'n\'t', 'cannot', 'without',  # Negación
    'but', 'however', 'although', 'though',  # Contraste
    'always', 'never', 'often', 'sometimes',  # Frecuencia
]

# Verificar cuáles están en STOP_WORDS
present_in_stopwords = [word for word in sentiment_relevant_stopwords if word in STOP_WORDS]
not_in_stopwords = [word for word in sentiment_relevant_stopwords if word not in STOP_WORDS]

print(f"Palabras relevantes que SÍ están en stopwords:")
print(sorted(present_in_stopwords))
print(f"\nPalabras relevantes que NO están en stopwords:")
print(sorted(not_in_stopwords))

Palabras relevantes que SÍ están en stopwords:
['although', 'always', 'but', 'cannot', 'however', "n't", 'neither', 'never', 'never', 'no', 'nobody', 'nor', 'not', 'nothing', 'nowhere', 'often', 'sometimes', 'though', 'without']

Palabras relevantes que NO están en stopwords:
[]


Aplicaria la funcion, con las palabras excluidas identificadas arriba, a mi corpus.

In [77]:
# Crear un set de stopwords personalizado excluyendo palabras relevantes
custom_stopwords = STOP_WORDS.copy()
for word in present_in_stopwords:
    custom_stopwords.discard(word)

# Aplicar filtrado de stopwords
def remove_stopwords(text_list):
    """Elimina stopwords excepto las relevantes para sentimiento"""
    return [word for word in text_list if word not in custom_stopwords]

# Aplicar a todas las reviews
df['review_process'] = df['review_process'].apply(remove_stopwords)
df.head()

,review,sentiment,sentiment_label,review_process
0,"This product is oil based, weird, and definite...",5.0,0,"[product, oil, based,, weird,, definitely, wor..."
1,I was shown an expensive one like this in Alab...,5.0,0,"[shown, expensive, like, alabama., fell, love,..."
2,I used to buy and use the original product. I ...,1.0,1,"[buy, use, original, product., small, it,, tho..."
3,This product was advertised as brand new. Ho...,1.0,1,"[product, advertised, brand, new., however,, b..."
4,this was a new edition to our daughter's body ...,5.0,0,"[new, edition, daughter's, body, sprays, loved..."


- Aplicar la funcion para eliminar tildes y simbolos. 
- Usa regex [^a-zA-Z] para eliminar cualquier caracter que no sea una letra

In [78]:

def clean_words(word_list):
    """Elimina tildes, símbolos y caracteres no alfabéticos de cada palabra en la lista"""
    cleaned = []
    for word in word_list:
        # Normalizar y eliminar tildes
        word_normalized = unicodedata.normalize('NFKD', word).encode('ascii', errors='ignore').decode('utf-8')
        # Eliminar cualquier caracter que no sea letra (mantener solo a-z)
        word_cleaned = re.sub(r'[^a-zA-Z]', '', word_normalized)
        # Agregar solo si queda algo después de limpiar
        if word_cleaned:
            cleaned.append(word_cleaned.lower())
    return cleaned

# Aplicar limpieza 
df['review_process'] = df['review_process'].apply(clean_words)
df.head()

,review,sentiment,sentiment_label,review_process
0,"This product is oil based, weird, and definite...",5.0,0,"[product, oil, based, weird, definitely, works..."
1,I was shown an expensive one like this in Alab...,5.0,0,"[shown, expensive, like, alabama, fell, love, ..."
2,I used to buy and use the original product. I ...,1.0,1,"[buy, use, original, product, small, it, thoug..."
3,This product was advertised as brand new. Ho...,1.0,1,"[product, advertised, brand, new, however, box..."
4,this was a new edition to our daughter's body ...,5.0,0,"[new, edition, daughters, body, sprays, loved,..."


In [79]:
# Comparar review original vs procesada
print('Review original: {}'.format(df['review'].values[0]))
print('Review procesada: {}'.format(df['review_process'].values[0]))

Review original: This product is oil based, weird, and definitely works. It takes longer to use than traditional nail polish, but I finally found a good technique. First, use the dropper to put the liquid all over your nails. Let it soak for about five minutes then wipe off. This will take off about half of your nail polish. Then put the liquid directly onto a paper towel (use liberally) and rub your nails like you would with acetone. Tada!
Review procesada: ['product', 'oil', 'based', 'weird', 'definitely', 'works', 'takes', 'longer', 'use', 'traditional', 'nail', 'polish', 'but', 'finally', 'found', 'good', 'technique', 'first', 'use', 'dropper', 'liquid', 'nails', 'let', 'soak', 'minutes', 'wipe', 'off', 'half', 'nail', 'polish', 'liquid', 'directly', 'paper', 'towel', 'use', 'liberally', 'rub', 'nails', 'like', 'acetone', 'tada']


### Limpio y elimino reviews vacias

In [80]:
# Filtrar reviews vacías
df = df[df['review_process'].apply(lambda x: len(x) > 0)]
df.reset_index(drop=True, inplace=True)

print(f"Reviews después de eliminar vacías: {len(df)}")
df.head()

Reviews después de eliminar vacías: 5990


,review,sentiment,sentiment_label,review_process
0,"This product is oil based, weird, and definite...",5.0,0,"[product, oil, based, weird, definitely, works..."
1,I was shown an expensive one like this in Alab...,5.0,0,"[shown, expensive, like, alabama, fell, love, ..."
2,I used to buy and use the original product. I ...,1.0,1,"[buy, use, original, product, small, it, thoug..."
3,This product was advertised as brand new. Ho...,1.0,1,"[product, advertised, brand, new, however, box..."
4,this was a new edition to our daughter's body ...,5.0,0,"[new, edition, daughters, body, sprays, loved,..."


### Lematización con spaCy - Sólo para usar en ML con TF-IDF: reduzco la dimensionalidad y ayudo al modelo a aprender mejor los datos, y aumento la coincidencia de palabras. 

In [81]:
# Crear lematizador
lemmatizer = WordNetLemmatizer()

def lemmatize_words(word_list):
    #Lematiza cada palabra 
    return [lemmatizer.lemmatize(word) for word in word_list]

# Aplicar lematización
df['review_process_lemma'] = df['review_process'].apply(lemmatize_words)
df.head()

,review,sentiment,sentiment_label,review_process,review_process_lemma
0,"This product is oil based, weird, and definite...",5.0,0,"[product, oil, based, weird, definitely, works...","[product, oil, based, weird, definitely, work,..."
1,I was shown an expensive one like this in Alab...,5.0,0,"[shown, expensive, like, alabama, fell, love, ...","[shown, expensive, like, alabama, fell, love, ..."
2,I used to buy and use the original product. I ...,1.0,1,"[buy, use, original, product, small, it, thoug...","[buy, use, original, product, small, it, thoug..."
3,This product was advertised as brand new. Ho...,1.0,1,"[product, advertised, brand, new, however, box...","[product, advertised, brand, new, however, box..."
4,this was a new edition to our daughter's body ...,5.0,0,"[new, edition, daughters, body, sprays, loved,...","[new, edition, daughter, body, spray, loved, s..."


In [ ]:
# Guardar el DataFrame procesado lemmatizado 
df.to_pickle('Data/df_beauty_balanced.pkl')
print(f"DataFrame guardado: {len(df)} reviews")

In [ ]:
# Mantener solo las columnas necesarias para el modelo
df = df[['review_process_lemma', 'sentiment_label']]
df.head()

,review_filtered,sentiment_label
0,"[product, oil, based, weird, definitely, works...",0
1,"[shown, expensive, like, alabama, fell, love, ...",0
2,"[buy, use, original, product, small, it, thoug...",1
3,"[product, advertised, brand, new, however, box...",1
4,"[new, edition, daughters, body, sprays, loved,...",0


In [ ]:
# Guardar el DataFrame procesado con solo las columnas necesarias
df.to_pickle('Data/df_beauty_preprocessed.pkl')
print(f"DataFrame guardado: {len(df)} reviews con columnas {list(df.columns)}")